# Hyperliquid HIP-4 Python Tutorial

[Hyperliquid Python SDK](https://github.com/hyperliquid-dex/hyperliquid-python-sdk) \
[HIP-4 docs](https://hyperliquid.gitbook.io/hyperliquid-docs/hyperliquid-improvement-proposals-hips/hip-4-outcome-markets)

## 1. Setup

In [ ]:
# pip install hyperliquid-python-sdk eth-account python-dotenv requests

In [ ]:
import os
import math
import requests
from dotenv import load_dotenv
import eth_account

from hyperliquid.info import Info
from hyperliquid.exchange import Exchange
from hyperliquid.utils import constants

## 2. Place the Bet

In [ ]:
load_dotenv()

BASE_URL = constants.MAINNET_API_URL    # https://api.hyperliquid.xyz
# BASE_URL = constants.TESTNET_API_URL  # https://api.hyperliquid-testnet.xyz

PRIVATE_KEY = os.environ["HL_SECRET_KEY"]
ACCOUNT_ADDRESS = os.environ["HL_ACCOUNT_ADDRESS"]

account = eth_account.Account.from_key(PRIVATE_KEY)
info = Info(BASE_URL, skip_ws=True)
hyperliquid = Exchange(account, BASE_URL, account_address=ACCOUNT_ADDRESS)

In [ ]:
# The SDK doesn't fetch HIP-4 outcomes at init, so we 
# register every live outcome side in the SDK's lookup maps.
# Coin string: "#N" where N = 10*outcome + side. Asset id: 100_000_000 + N.

def register_hip4_outcomes(meta, *clients):
    for o in meta["outcomes"]:
        for side in (0, 1):
            coin = f"#{10 * o['outcome'] + side}"
            asset_id = 100_000_000 + 10 * o["outcome"] + side
            for client in clients:
                client.coin_to_asset[coin] = asset_id
                client.name_to_coin[coin] = coin

meta = requests.post(f"{BASE_URL}/info", json={"type": "outcomeMeta"}).json()
register_hip4_outcomes(meta, info, hyperliquid.info)


In [ ]:
COIN = "#600"           # YES side of outcome 60
IS_BUY = True           # buying = going long this outcome
PRICE = 0.65            # limit price per YES contract (between 0.001 and 0.999)
TARGET_AMOUNT = 12      # USDH targeted for this bet (minimum notional must be 10)
ORDER_TYPE = {"limit": {"tif": "Gtc"}}    # "good-till-cancelled" limit order

result = hyperliquid.order(
    name=COIN,
    is_buy=IS_BUY,
    sz=math.ceil(TARGET_AMOUNT / PRICE),   # contracts; Hyperliquid requires whole integers
    limit_px=PRICE,
    order_type=ORDER_TYPE,
)
result

## 3. Where `COIN` comes from

In [ ]:
meta = requests.post(f"{BASE_URL}/info", json={"type": "outcomeMeta"}).json()

In [ ]:
meta["questions"]

In [ ]:
meta["outcomes"]

In [ ]:
outcome_id = 60
yes_coin = f"YES is #{10 * outcome_id}"
no_coin  = f"No is #{10 * outcome_id + 1}"
print(yes_coin, no_coin)

In [ ]:
def parse(desc):
    return dict(kv.split(":", 1) if ":" in kv else (kv, True) for kv in desc.split("|"))

def fmt_expiry(s):
    return f"{s[:4]}-{s[4:6]}-{s[6:8]} {s[9:11]}:{s[11:13]} UTC"

parent = {o: q for q in meta["questions"] for o in q["namedOutcomes"] + [q["fallbackOutcome"]]}

for o in meta["outcomes"]:
    f = parse(o["description"])

    if f.get("class") == "priceBinary":
        label = f"{f['underlying']} above ${int(f['targetPrice']):,} at {fmt_expiry(f['expiry'])}"
    elif "other" in f:
        label = f"fallback for question {parent.get(o['outcome'], {}).get('question', '?')}"
    elif "index" in f:
        q = parent.get(o["outcome"])
        qf = parse(q["description"])
        thresholds = [int(t) for t in qf["priceThresholds"].split(",")]
        idx = int(f["index"])
        if idx == 0:
            range_str = f"below ${thresholds[0]:,}"
        elif idx >= len(thresholds):
            range_str = f"above ${thresholds[-1]:,}"
        else:
            range_str = f"between ${thresholds[idx-1]:,} and ${thresholds[idx]:,}"
        label = f"{qf['underlying']} {range_str} at {fmt_expiry(qf['expiry'])}"
    else:
        label = o["description"]

    print(f"outcome {o['outcome']:>3}  YES=#{10*o['outcome']}  NO=#{10*o['outcome']+1}  {label}")

## 4. `PRICE` vs Order Book

In [ ]:
def book_snapshot(book):
    bids, asks = book["levels"]
    best_bid = float(bids[0]["px"])
    best_ask = float(asks[0]["px"])
    return {
        "best_bid": best_bid,
        "best_ask": best_ask,
        "mid": (best_bid + best_ask) / 2,
        "spread": best_ask - best_bid,
    }

book = requests.post(f"{BASE_URL}/info", json={"type": "l2Book", "coin": COIN}).json()
bids, asks = book["levels"]

print("asks:")
for lvl in asks[:5][::-1]:
    print(f"  {lvl['px']:>8}  size {lvl['sz']}")
print("bids:")
for lvl in bids[:5]:
    print(f"  {lvl['px']:>8}  size {lvl['sz']}")

print()
book_snapshot(book)


## 5. Where `SIZE` comes from

In [ ]:
SIZE = math.ceil(TARGET_AMOUNT / PRICE)
NOTIONAL_AMOUNT = SIZE * PRICE

print(f"TARGET_AMOUNT   = {TARGET_AMOUNT} USDH")
print(f"SIZE            = {SIZE} contracts")
print(f"NOTIONAL_AMOUNT = {NOTIONAL_AMOUNT:.2f} USDH (actual cost)")


## 6. Where `ORDER_TYPE` comes from

`tif` (time in force) controls behavior:
- `Gtc` — rests in the book until filled or cancelled.
- `Ioc` — fills what it can at PRICE or better, cancels the rest (set PRICE close to 0.999 for a buy and this acts like a market order).
- `Alo` — post-only; rejected if it would cross the spread.

In [ ]:
ORDER_TYPE = {"limit": {"tif": "Gtc"}}
# ORDER_TYPE = {"limit": {"tif": "Ioc"}}
# ORDER_TYPE = {"limit": {"tif": "Alo"}}

## 7. Where USDH (the collateral) comes from

In [ ]:
spot = info.spot_user_state(ACCOUNT_ADDRESS)
balances = {b["coin"]: float(b["total"]) for b in spot["balances"]}
print(f"USDH: {balances.get('USDH', 0):.4f}")
print(f"USDC: {balances.get('USDC', 0):.4f}")

## 8. API keys and Outcome registration

In [ ]:
load_dotenv()

BASE_URL = constants.MAINNET_API_URL    # https://api.hyperliquid.xyz
# BASE_URL = constants.TESTNET_API_URL  # https://api.hyperliquid-testnet.xyz

PRIVATE_KEY = os.environ["HL_SECRET_KEY"]
ACCOUNT_ADDRESS = os.environ["HL_ACCOUNT_ADDRESS"]

account = eth_account.Account.from_key(PRIVATE_KEY)
info = Info(BASE_URL, skip_ws=True)
hyperliquid = Exchange(account, BASE_URL, account_address=ACCOUNT_ADDRESS)

In [ ]:
# The SDK doesn't fetch HIP-4 outcomes at init, so we 
# register every live outcome side in the SDK's lookup maps.
# Coin string: "#N" where N = 10*outcome + side. Asset id: 100_000_000 + N.

def register_hip4_outcomes(meta, *clients):
    for o in meta["outcomes"]:
        for side in (0, 1):
            coin = f"#{10 * o['outcome'] + side}"
            asset_id = 100_000_000 + 10 * o["outcome"] + side
            for client in clients:
                client.coin_to_asset[coin] = asset_id
                client.name_to_coin[coin] = coin

meta = requests.post(f"{BASE_URL}/info", json={"type": "outcomeMeta"}).json()
register_hip4_outcomes(meta, info, hyperliquid.info)

## 9. After the trade

In [ ]:
# All open HIP-4 orders on the account.
print(info.open_orders(ACCOUNT_ADDRESS))

In [ ]:
# Cancel a single order by its oid (uses the order id from the bet cell's result).
oid = result["response"]["data"]["statuses"][0].get("resting", {}).get("oid")
if oid:
    print(hyperliquid.cancel(name=COIN, oid=oid))

In [ ]:
# Cancel all open HIP-4 order on the account.
hip4_orders = [o for o in info.open_orders(ACCOUNT_ADDRESS) if o["coin"].startswith("#")]
if hip4_orders:
    cancels = [{"coin": o["coin"], "oid": o["oid"]} for o in hip4_orders]
    print(hyperliquid.bulk_cancel(cancels))

In [ ]:
# HIP-4 holdings live in `spotClearinghouseState` under coin names prefixed with `+`
# (e.g. `+600` is the spot-balance form of `#600`).

spot = info.spot_user_state(ACCOUNT_ADDRESS)
spot

In [ ]:
# Close a HIP-4 position with an opposite-side IOC order at the current best bid.

book = requests.post(f"{BASE_URL}/info", json={"type": "l2Book", "coin": COIN}).json()
best_bid = float(book["levels"][0][0]["px"])
size = int(float(next(b["total"] for b in spot["balances"] if b["coin"] == "+" + COIN[1:])))

hyperliquid.order(
    name=COIN,
    is_buy=False,
    sz=size,
    limit_px=best_bid,
    order_type={"limit": {"tif": "Ioc"}},
)